In [ ]:
from __future__ import annotations

import argparse, csv, os, pathlib, re, sys, time
from dataclasses import dataclass
from typing import Iterator, List, Tuple

import pandas as pd
from bs4 import BeautifulSoup  # pip install beautifulsoup4
from selenium import webdriver  # pip install selenium~=4.21.0
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import urllib.parse

###############################################################################
# 0. Config & helpers
###############################################################################

BLOCKWORDS = {
    "협찬", "체험단", "서포터즈", "공구", "원고료", "지원받아", "스폰", "서평", "AD", "ad",
    "광고", "홍보", "리뷰어", "샘플", "provided", "sponsored", "gifted",
}

# CSS 선택자 (Selectors)
BLOG_LINK_SELECTOR = "a.desc_inner"
PUBLISH_DATE_SEL   = "span.se_publishDate, span.date"
CONTENT_SEL        = "div.se-main-container, #postViewArea"

try:
    SCRIPT_DIR = pathlib.Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = pathlib.Path.cwd()


def sanitize(text: str) -> str:
    """공백을 하나로 줄이고 제어 문자를 제거하여 텍스트를 정리합니다."""
    clean = re.sub(r"[\x00-\x1f\u2028\u2029]+", " ", text)
    clean = re.sub(r"\s+", " ", clean).strip()
    return clean


def is_sponsored(text: str, allow: bool = False) -> bool:
    """텍스트에 협찬/광고 관련 단어가 포함되어 있는지 확인합니다."""
    if allow:
        return False
    lower = text.lower()
    return any(word.lower() in lower for word in BLOCKWORDS)


###############################################################################
# 1. Selenium boilerplate
###############################################################################

def make_driver(headless: bool = True) -> webdriver.Chrome:
    """Selenium Chrome 웹 드라이버 인스턴스를 생성합니다."""
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--lang=ko-KR")
    options.add_argument("--window-size=1280,1024")
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


###############################################################################
# 2. Core crawler
###############################################################################

@dataclass
class Review:
    """수집된 리뷰 데이터를 저장하기 위한 데이터 클래스입니다."""
    search_keyword: str
    place_name: str
    address: str
    url: str
    date: str
    text: str

    def as_tuple(self) -> Tuple[str, str, str, str, str, str]:
        """데이터를 CSV 저장을 위해 튜플 형태로 변환합니다."""
        return (
            self.search_keyword,
            self.place_name,
            self.address,
            self.url,
            self.date,
            self.text,
        )

# ################## 이 함수를 아래 코드로 교체해주세요 ##################
def search_blog_links(driver: webdriver.Chrome, keyword: str, max_links: int) -> List[str]:
    """네이버 블로그 검색을 URL로 직접 요청하고 링크를 수집합니다."""
    links: list[str] = []
    print(f"   - '{keyword}' 검색 시작...")

    # 1. 검색어를 URL 인코딩하여 검색 결과 페이지로 바로 이동
    encoded_keyword = urllib.parse.quote(keyword)
    
    # 페이지 번호를 바꿔가며 URL을 직접 호출할 것이므로 초기 페이지는 1로 설정
    current_page = 1
    
    while True:
        # 페이지 번호를 포함한 검색 URL 생성
        search_url = f"https://section.blog.naver.com/Search/Post.naver?pageNo={current_page}&rangeType=ALL&orderBy=sim&keyword={encoded_keyword}"
        
        try:
            driver.get(search_url)
            # 페이지가 로드될 시간을 줍니다.
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div.area_list_search"))
            )
        except Exception as e:
            print(f"[경고] 검색 결과 페이지({search_url}) 로딩에 실패했습니다: {e}", file=sys.stderr)
            break

        # 2. 현재 페이지의 모든 블로그 링크 수집
        elements = driver.find_elements(By.CSS_SELECTOR, BLOG_LINK_SELECTOR)
        if not elements:
            print("[정보] 현재 페이지에서 더 이상 링크를 찾지 못해 중단합니다.")
            break

        new_links_found_on_page = False
        for a in elements:
            url = a.get_attribute("href")
            if url and url not in links:
                links.append(url)
                new_links_found_on_page = True

        print(f"     - {current_page} 페이지에서 {len(elements)}개 링크 확인, 총 {len(links)}개 수집됨.")

        # 3. 목표 수량 도달 시 중단
        if len(links) >= max_links:
            print(f"[정보] 목표 수량({max_links}개) 이상의 링크를 탐색하여 중단합니다.")
            break
        
        # 4. 다음 페이지로 이동
        try:
            # 현재 페이지네이션 블록에서 다음 번호로 이동
            pagination = driver.find_element(By.CLASS_NAME, "pagination")
            current_page_element = pagination.find_element(By.CSS_SELECTOR, "strong.current")
            
            # 다음 페이지 번호(<a> 태그)가 있는지 확인
            next_page_link = current_page_element.find_element(By.XPATH, "./parent::span/following-sibling::span/a")
            current_page += 1
            
        except NoSuchElementException:
            # 페이지 번호가 더 없다면 '다음' 버튼으로 페이지 블록을 넘김
            try:
                next_block_button = driver.find_element(By.CSS_SELECTOR, "a.button_next")
                # '다음' 버튼이 비활성화(disabled) 상태가 아니어야 함
                if "disabled" in next_block_button.get_attribute("class"):
                    print("[정보] 마지막 페이지 블록에 도달하여 수집을 중단합니다.")
                    break
                current_page += 1
            except NoSuchElementException:
                print("[정보] 모든 페이지를 확인하여 수집을 중단합니다.")
                break

    return links[:max_links]

def extract_post(driver: webdriver.Chrome, url: str) -> tuple[str | None, str | None]:
    """개별 블로그 포스트 URL에 접속하여 날짜와 본문 텍스트를 추출합니다."""
    try:
        driver.get(url)
        time.sleep(1.4)
        if "blog.naver.com" in driver.current_url and "PostView.naver" not in driver.current_url:
            try:
                driver.switch_to.frame("mainFrame")
            except Exception:
                pass

        soup = BeautifulSoup(driver.page_source, "html.parser")
        date_elt = soup.select_one(PUBLISH_DATE_SEL)
        date = sanitize(date_elt.get_text()) if date_elt else ""
        content_elt = soup.select_one(CONTENT_SEL)
        text = sanitize(content_elt.get_text(separator=" ")) if content_elt else ""
        driver.switch_to.default_content()
        return date, text
    except Exception as e:
        print(f"[경고] 포스트 추출 중 오류 발생: {e} – {url}", file=sys.stderr)
        return None, None


def crawl_reviews(
    keyword: str,
    place_name: str,
    address: str,
    *,
    search_limit: int,
    allow_sponsored: bool = False,
    driver: webdriver.Chrome | None = None,
    done_urls: set[str] | None = None,
) -> Iterator[Review]:
    """주어진 키워드로 리뷰를 크롤링합니다."""
    close_driver = False
    if driver is None:
        driver = make_driver(headless=True)
        close_driver = True
    
    if done_urls is None:
        done_urls = set()

    try:
        links = search_blog_links(driver, keyword, max_links=search_limit)
        print(f"   - 총 {len(links)}개 링크 수집 (최대 {search_limit}개까지 확인)")

        for i, link in enumerate(links):
            if link in done_urls:
                continue

            print(f"     - 포스트 {i+1}/{len(links)} 추출 중...")
            date, text = extract_post(driver, link)
            if not (date and text):
                continue
            if is_sponsored(text, allow=allow_sponsored):
                print(f"       - 광고/협찬 포스트 제외: {link}")
                continue
            
            yield Review(keyword, place_name, address, link, date, text)
    finally:
        if close_driver:
            driver.quit()


###############################################################################
# 3. CLI batch pipeline
###############################################################################

def open_csv_writer(path: pathlib.Path, *, resume: bool) -> tuple[csv.writer, any]:
    mode = "a" if resume and path.exists() else "w"
    f = open(path, mode, newline="", encoding="utf-8-sig")
    writer = csv.writer(f)
    if mode == "w" or os.stat(path).st_size == 0:
        writer.writerow(["search_keyword", "place_name", "address", "url", "date", "review_text"])
        f.flush()
    return writer, f


def load_done_info(path: pathlib.Path) -> tuple[dict[str, int], set[str]]:
    """CSV에서 place_name별 리뷰 개수와 저장된 URL 세트를 반환합니다."""
    if not path.exists() or os.stat(path).st_size == 0:
        return {}, set()
    try:
        df = pd.read_csv(path, usecols=["place_name", "url"])
        counts = df["place_name"].value_counts().to_dict()
        urls = set(df["url"].dropna())
        return counts, urls
    except (ValueError, KeyError) as e:
        print(f"[경고] CSV 파일('{path}')을 읽는 중 오류: {e}", file=sys.stderr)
        print("[경고] 'place_name'과 'url' 컬럼이 있는지 확인하세요. 이어쓰기를 비활성화합니다.", file=sys.stderr)
        return {}, set()


# ################## 이 함수가 수정되었습니다 ##################
def run_batch(
    input_df: pd.DataFrame,
    output_csv: pathlib.Path,
    *,
    max_posts: int = 3,
    rate_limit: float = 1.0,
    allow_sponsored: bool = False,
    resume: bool = True,
):
    """선택된 데이터프레임에 대해 크롤링을 수행합니다."""
    df = input_df.copy()
    df["address"] = df["addr1"].fillna("") + " " + df["addr2"].fillna("")

    # CSV 에서 이미 수집된 정보 불러오기
    done_counts, done_urls = load_done_info(output_csv) if resume else ({}, set())
    # 실행 중 누적 카운트를 위해 초기값 복사
    local_counts: dict[str, int] = dict(done_counts)

    writer, fh = open_csv_writer(output_csv, resume=resume)
    driver = make_driver(headless=True)

    SKIP_THRESHOLD = 15

    try:
        for i, row in df.iterrows():
            place_name = row["title"]
            already = local_counts.get(place_name, 0)

            # ─── 새 로직: 이미 15개 이상이면 스킵 ───
            if already >= SKIP_THRESHOLD:
                print(f"[{i+1}/{len(df)}] ✅ 스킵: '{place_name}' (이미 {already}개 수집, 기준 {SKIP_THRESHOLD}개 이상)")
                continue

            # max_posts 기준 스킵 (기본 동작)
            if already >= max_posts:
                print(f"[{i+1}/{len(df)}] ✅ 건너뛰기: {place_name} (이미 {already}개 수집 완료)")
                continue

            needed = max_posts - already
            print(f"[{i+1}/{len(df)}] ⏩ {place_name} (수집된 리뷰: {already}개, 필요한 리뷰: {needed}개)")

            search_limit = max(needed * 5, 100)
            newly_added = 0

            for rev in crawl_reviews(
                place_name, place_name, sanitize(row["address"]),
                search_limit=search_limit,
                allow_sponsored=allow_sponsored,
                driver=driver,
                done_urls=done_urls,
            ):
                writer.writerow(rev.as_tuple()); fh.flush()
                done_urls.add(rev.url)
                newly_added += 1
                local_counts[place_name] = already + newly_added

                if newly_added >= needed:
                    break

            if newly_added == 0 and already == 0:
                writer.writerow([place_name, place_name, sanitize(row["address"]), "(no review)", "", ""])
                fh.flush()

            time.sleep(rate_limit)

    finally:
        driver.quit()
        fh.close()
        print(f"✅ 작업 완료 — 데이터 저장 위치: {output_csv}")



###############################################################################
# ################## 이 부분이 수정되었습니다 ##################
if __name__ == "__main__":
    # --- 파일 이름 설정 ---
    input_excel_file = '전국_시도별/서울특별시.xlsx'
    output_csv_file = '전국_그룹분리_csv/서울2.csv'

    print("크롤링을 시작합니다...")
    print(f"입력 파일: {input_excel_file}")
    print(f"출력 파일: {output_csv_file}")
    
    # 1. 전체 엑셀 파일을 읽어옵니다.
    try:
        full_df = pd.read_excel(input_excel_file)
    except FileNotFoundError:
        print(f"[오류] 입력 엑셀 파일 '{input_excel_file}'을 찾을 수 없습니다. 경로를 확인해주세요.")
        sys.exit(1)

    # 파이썬은 0부터 숫자를 셉니다. (엑셀 1행 = 0, 엑셀 2행 = 1)
    # 예시 1: 5번 행부터 10번 행까지 (엑셀 기준 6행 ~ 11행)
    start_row = 50
    end_row = 100
        # end_row가 None인 경우, 끝까지 슬라이싱합니다.
    end_slice = end_row + 1 if end_row is not None else None
    selected_df = full_df.iloc[start_row:end_slice]

    print(f"INFO: 총 {len(full_df)}개 중 선택된 {len(selected_df)}개 행에 대해 크롤링을 진행합니다.")

    # 3. 선택된 데이터만 run_batch 함수에 전달합니다.
    run_batch(
        input_df=selected_df,
        output_csv=pathlib.Path(output_csv_file),
        max_posts=50,
        rate_limit=1.0,
        allow_sponsored=False,
        resume=True,
    )

크롤링을 시작합니다...
입력 파일: 전국_시도별/서울특별시.xlsx
출력 파일: 전국_그룹분리_csv/서울2.csv
INFO: 총 7334개 중 선택된 51개 행에 대해 크롤링을 진행합니다.
[51/51] ✅ 스킵: '감미옥' (이미 50개 수집, 기준 15개 이상)
[52/51] ✅ 스킵: '감포면옥' (이미 50개 수집, 기준 15개 이상)
[53/51] ✅ 스킵: '값진식육' (이미 50개 수집, 기준 15개 이상)
[54/51] ✅ 스킵: '갓덴스시 강남' (이미 50개 수집, 기준 15개 이상)
[55/51] ✅ 스킵: '갓잇' (이미 50개 수집, 기준 15개 이상)
[56/51] ✅ 스킵: '갓포아키 도산공원점' (이미 50개 수집, 기준 15개 이상)
[57/51] ✅ 스킵: '갓포준 x 에스테반' (이미 43개 수집, 기준 15개 이상)
[58/51] ✅ 스킵: '강가네돌솥밥추어탕' (이미 50개 수집, 기준 15개 이상)
[59/51] ✅ 스킵: '강강술래 신림동' (이미 50개 수집, 기준 15개 이상)
[60/51] ✅ 스킵: '강남' (이미 50개 수집, 기준 15개 이상)
[61/51] ✅ 스킵: '강남 마이스 관광특구' (이미 26개 수집, 기준 15개 이상)
[62/51] ✅ 스킵: '강남고속터미널 혼수상가' (이미 50개 수집, 기준 15개 이상)
[63/51] ✅ 스킵: '강남목장' (이미 50개 수집, 기준 15개 이상)
[64/51] ✅ 스킵: '강남문화원' (이미 50개 수집, 기준 15개 이상)
[65/51] ✅ 스킵: '강남역 케미스트릿 페스티벌' (이미 31개 수집, 기준 15개 이상)
[66/51] ✅ 스킵: '강남역데이원의원' (이미 50개 수집, 기준 15개 이상)
[67/51] ⏩ 강남중고명품 아마레스 (수집된 리뷰: 1개, 필요한 리뷰: 49개)
   - '강남중고명품 아마레스' 검색 시작...
     - 1 페이지에서 7개 링크 확인, 총 7개 수집됨.
[정보] 모든 페이지를 확인하여 수집을 중단합니다.

In [ ]:
# from __future__ import annotations

# import argparse, csv, os, pathlib, re, sys, time
# from dataclasses import dataclass
# from typing import Iterator, List, Tuple

# import pandas as pd
# from bs4 import BeautifulSoup  # pip install beautifulsoup4
# from selenium import webdriver  # pip install selenium~=4.21.0
# from selenium.common.exceptions import NoSuchElementException
# from selenium.webdriver.common.by import By
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.remote.webelement import WebElement
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC


# ###############################################################################
# # 0. Config & helpers
# ###############################################################################

# BLOCKWORDS = {
#     "협찬", "체험단", "서포터즈", "공구", "원고료", "지원받아", "스폰", "서평", "AD", "ad",
#     "광고", "홍보", "리뷰어", "샘플", "provided", "sponsored", "gifted",
# }

# # CSS 선택자 (Selectors)
# BLOG_LINK_SELECTOR = "a.desc_inner"
# PUBLISH_DATE_SEL   = "span.se_publishDate, span.date"
# CONTENT_SEL        = "div.se-main-container, #postViewArea"

# try:
#     SCRIPT_DIR = pathlib.Path(__file__).resolve().parent
# except NameError:
#     SCRIPT_DIR = pathlib.Path.cwd()


# def sanitize(text: str) -> str:
#     """공백을 하나로 줄이고 제어 문자를 제거하여 텍스트를 정리합니다."""
#     clean = re.sub(r"[\x00-\x1f\u2028\u2029]+", " ", text)
#     clean = re.sub(r"\s+", " ", clean).strip()
#     return clean


# def is_sponsored(text: str, allow: bool = False) -> bool:
#     """텍스트에 협찬/광고 관련 단어가 포함되어 있는지 확인합니다."""
#     if allow:
#         return False
#     lower = text.lower()
#     return any(word.lower() in lower for word in BLOCKWORDS)


# def locate_xlsx(path_like: str | pathlib.Path) -> pathlib.Path:
#     """주어진 경로에서 엑셀 파일을 찾거나, 없으면 프로젝트 루트에서 검색합니다."""
#     p = pathlib.Path(path_like)
#     if p.expanduser().is_absolute():
#         return p
#     if p.exists():
#         return p
#     project_root = SCRIPT_DIR.parent
#     hits = list(project_root.rglob(p.name))
#     if hits:
#         print(f"[정보] 엑셀 파일 위치 확인 → {hits[0].relative_to(project_root)}")
#         return hits[0]
#     raise FileNotFoundError(f"엑셀 파일 '{p}'를 찾을 수 없습니다 (검색 경로: {project_root})")


# ###############################################################################
# # 1. Selenium boilerplate
# ###############################################################################

# def make_driver(headless: bool = True) -> webdriver.Chrome:
#     """Selenium Chrome 웹 드라이버 인스턴스를 생성합니다."""
#     options = Options()
#     if headless:
#         options.add_argument("--headless=new")
#     options.add_argument("--disable-gpu")
#     options.add_argument("--no-sandbox")
#     options.add_argument("--lang=ko-KR")
#     options.add_argument("--window-size=1280,1024")
#     driver = webdriver.Chrome(options=options)
#     driver.implicitly_wait(3)
#     return driver


# ###############################################################################
# # 2. Core crawler
# ###############################################################################

# @dataclass
# class Review:
#     """수집된 리뷰 데이터를 저장하기 위한 데이터 클래스입니다."""
#     search_keyword: str
#     place_name: str
#     address: str
#     date: str
#     text: str

#     def as_tuple(self) -> Tuple[str, str, str, str, str]:
#         """데이터를 CSV 저장을 위해 튜플 형태로 변환합니다."""
#         return (
#             self.search_keyword,
#             self.place_name,
#             self.address,
#             self.date,
#             self.text,
#         )


# def search_blog_links(driver: webdriver.Chrome, keyword: str, max_links: int) -> List[str]:
#     """네이버 블로그의 2단계 페이지네이션을 처리하며 링크를 수집합니다."""
#     links: list[str] = []
#     print(f"  - 블로그 홈에서 '{keyword}' 검색 시작...")
#     driver.get("https://section.blog.naver.com/")
#     time.sleep(1.5)

#     try:
#         search_box = WebDriverWait(driver, 10).until(
#             EC.presence_of_element_located((By.CSS_SELECTOR, "input.textbox[name='sectionBlogQuery']"))
#         )
#         search_box.send_keys(keyword)
#         search_box.submit()
#         time.sleep(1.5)
#     except Exception as e:
#         print(f"[경고] 블로그 섹션 페이지에서 검색을 수행하지 못했습니다: {e}", file=sys.stderr)
#         return []

#     while True:
#         elements = driver.find_elements(By.CSS_SELECTOR, BLOG_LINK_SELECTOR)
#         new_links = False
#         for a in elements:
#             url = a.get_attribute("href")
#             if url and url not in links:
#                 links.append(url)
#                 new_links = True

#         if len(links) >= max_links:
#             print(f"[정보] 목표 수량({max_links}개) 이상의 링크를 수집하여 중단합니다.")
#             break
#         if not new_links and links:
#             print("[정보] 현재 페이지에서 더 이상 새로운 링크를 찾지 못해 중단합니다.")
#             break

#         try:
#             pagination = driver.find_element(By.CLASS_NAME, "pagination")
#             current = pagination.find_element(By.CSS_SELECTOR, "strong")
#             nxt = current.find_element(By.XPATH, "./parent::span/following-sibling::span/a")
#             print(f"    - {nxt.text} 페이지로 이동...")
#             driver.execute_script("arguments[0].click();", nxt)
#             time.sleep(1.2)
#         except NoSuchElementException:
#             try:
#                 btn = driver.find_element(By.CSS_SELECTOR, "a.button_next")
#                 print("    - '다음' 그룹 페이지로 이동...")
#                 driver.execute_script("arguments[0].click();", btn)
#                 time.sleep(1.2)
#             except NoSuchElementException:
#                 print("[정보] 모든 페이지를 확인하여 수집을 중단합니다.")
#                 break
#         except Exception as e:
#             print(f"[경고] 페이지 이동 중 오류 발생: {e}", file=sys.stderr)
#             break

#     return links[:max_links]


# def extract_post(driver: webdriver.Chrome, url: str) -> tuple[str | None, str | None]:
#     """개별 블로그 포스트 URL에 접속하여 날짜와 본문 텍스트를 추출합니다."""
#     try:
#         driver.get(url)
#         time.sleep(1.4)
#         if "blog.naver.com" in driver.current_url and "PostView.naver" not in driver.current_url:
#             try:
#                 driver.switch_to.frame("mainFrame")
#             except Exception:
#                 pass

#         soup = BeautifulSoup(driver.page_source, "html.parser")
#         date_elt = soup.select_one(PUBLISH_DATE_SEL)
#         date = sanitize(date_elt.get_text()) if date_elt else ""
#         content_elt = soup.select_one(CONTENT_SEL)
#         text = sanitize(content_elt.get_text(separator=" ")) if content_elt else ""
#         driver.switch_to.default_content()
#         return date, text
#     except Exception as e:
#         print(f"[경고] 포스트 추출 중 오류 발생: {e} – {url}", file=sys.stderr)
#         return None, None


# def crawl_reviews(
#     keyword: str,
#     place_name: str,
#     address: str,
#     *,
#     max_posts: int = 5,
#     allow_sponsored: bool = False,
#     driver: webdriver.Chrome | None = None,
#     start_index: int = 0,
# ) -> Iterator[Review]:
#     """주어진 키워드로 리뷰를 크롤링, start_index부터 이어서."""
#     close_driver = False
#     if driver is None:
#         driver = make_driver(headless=True)
#         close_driver = True

#     try:
#         links = search_blog_links(driver, keyword, max_links=max_posts)
#         print(f"  - 총 {len(links)}개 링크 수집 (건너뛸 개수: {start_index})")
#         for i, link in enumerate(links):
#             if i < start_index:
#                 continue
#             print(f"    - 포스트 {i+1}/{len(links)} 추출 중...")
#             date, text = extract_post(driver, link)
#             if not (date and text):
#                 continue
#             if is_sponsored(text, allow=allow_sponsored):
#                 print(f"      - 광고/협찬 포스트 제외: {link}")
#                 continue
#             yield Review(keyword, place_name, address, date, text)
#     finally:
#         if close_driver:
#             driver.quit()


# ###############################################################################
# # 3. CLI batch pipeline
# ###############################################################################

# def open_csv_writer(path: pathlib.Path, *, resume: bool) -> tuple[csv.writer, any]:
#     mode = "a" if resume and path.exists() else "w"
#     f = open(path, mode, newline="", encoding="utf-8-sig")
#     writer = csv.writer(f)
#     if mode == "w" or os.stat(path).st_size == 0:
#         writer.writerow(["search_keyword", "place_name", "address", "date", "review_text"])
#         f.flush()
#     return writer, f


# def load_done_counts(path: pathlib.Path) -> dict[str, int]:
#     """
#     CSV에서 place_name별로 저장된 리뷰 개수를 반환합니다.
#     """
#     if not path.exists():
#         return {}
#     try:
#         df = pd.read_csv(path, usecols=["place_name"])
#         return df["place_name"].value_counts().to_dict()
#     except Exception:
#         return {}


# def run_batch(
#     xlsx: pathlib.Path | str,
#     output_csv: pathlib.Path,
#     *,
#     max_posts: int = 3,
#     rate_limit: float = 1.0,
#     allow_sponsored: bool = False,
#     resume: bool = True,
# ):
#     xlsx_path = locate_xlsx(xlsx)
#     df = pd.read_excel(xlsx_path)
#     df["address"] = df["addr1"].fillna("") + " " + df["addr2"].fillna("")

#     done_counts = load_done_counts(output_csv) if resume else {}
#     if done_counts:
#         total_done_places = sum(1 for v in done_counts.values() if v >= max_posts)
#         print(f"[정보] 이어쓰기 모드 — {len(done_counts)}개 장소 중 {total_done_places}곳이 목표치({max_posts}) 이상 처리됨")

#     writer, fh = open_csv_writer(output_csv, resume=resume)
#     driver = make_driver(headless=True)

#     try:
#         for i, row in df.iterrows():
#             place_name = row["title"]
#             already = done_counts.get(place_name, 0)
#             if already >= max_posts:
#                 continue

#             search_keyword = place_name
#             address = sanitize(row["address"])
#             print(f"[{i+1}/{len(df)}] ⏩ {search_keyword} (이미 {already}개 수집됨)")

#             new_rows = 0
#             for rev in crawl_reviews(
#                 search_keyword,
#                 place_name,
#                 address,
#                 max_posts=max_posts,
#                 allow_sponsored=allow_sponsored,
#                 driver=driver,
#                 start_index=already,
#             ):
#                 writer.writerow(rev.as_tuple())
#                 fh.flush()
#                 new_rows += 1

#             if new_rows == 0 and already == 0:
#                 writer.writerow([search_keyword, place_name, address, "", "(no review)"])
#                 fh.flush()

#             time.sleep(rate_limit)
#     finally:
#         driver.quit()
#         fh.close()
#         print(f"✅ 작업 완료 — 데이터 저장 위치: {output_csv}")


# ###############################################################################
# if __name__ == "__main__":
#     # --- 여기서 파일 이름을 설정하세요 ---
#     input_excel_file = '전국_그룹분리/서울.xlsx'      # 👈 입력할 엑셀 파일 이름
#     output_csv_file = '전국_그룹분리_csv/서울.csv'    # 👈 저장될 CSV 파일 이름

#     print(f"크롤링을 시작합니다...")
#     print(f"입력 파일: {input_excel_file}")
#     print(f"출력 파일: {output_csv_file}")

#     run_batch(
#         xlsx=input_excel_file,
#         output_csv=pathlib.Path(output_csv_file),
#         max_posts=100,
#         rate_limit=1.0,
#         allow_sponsored=False,
#         resume=True,
#     )


#### 지역별로

In [ ]:
from pathlib import Path
import pandas as pd, csv, re

# 파일 경로 설정
FILE1 = Path(r"N:\개인\Seoul_Strolling_Adventure\크롤링\전국_그룹분리_csv\인천.csv")
FILE2 = Path(r"N:\개인\Seoul_Strolling_Adventure\크롤링\전국_그룹분리_csv\인천2.csv")
OUTFILE = FILE1.with_stem("인천")

# 공통 열 지정
COLS = ["search_keyword", "place_name", "address", "url" ,"date", "review_text"]

# 두 파일 읽기
df1 = pd.read_csv(FILE1, names=COLS, header=0, quoting=csv.QUOTE_MINIMAL, on_bad_lines="skip", encoding="utf-8")
df2 = pd.read_csv(FILE2, names=COLS, header=0, quoting=csv.QUOTE_MINIMAL, on_bad_lines="skip", encoding="utf-8")

# 병합
df = pd.concat([df1, df2], ignore_index=True)

# 리뷰 정규화
df["review_text_norm"] = (
    df["review_text"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# 중복 제거
KEY_COLS = ["review_text_norm"]
dup_mask = df.duplicated(subset=KEY_COLS, keep="first")
print(f"중복 제거: {dup_mask.sum()}행 삭제")
df_clean = df.loc[~dup_mask].drop(columns=["review_text_norm"])

# 🔽 address 기준 내림차순 정렬
df_clean_sorted = df_clean.sort_values(by="address", ascending=False)

# 저장
df_clean_sorted.to_csv(OUTFILE, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL)
print("최종 저장 완료 →", OUTFILE)


In [ ]:
# #!/usr/bin/env python3
# # -*- coding: utf-8 -*-
# """
# naver_reviews_clean.py
# ────────────────────────────────────────────────────────
# CSV 깨짐-방지 + 중복 제거 스크립트 (Windows 환경 기준)
# """

# from pathlib import Path
# import csv, pandas as pd, re

# # ───────────────────────────────────────────────────────
# # 1. 파일 경로
# # ───────────────────────────────────────────────────────
# FILE = Path(r"N:\개인\Seoul_Strolling_Adventure\크롤링\전국_그룹분리_csv\인천.csv")

# # ───────────────────────────────────────────────────────
# # 2. CSV 읽기 – 파서 오류 건너뛰기
# #    (필드 수 안 맞는 행 on_bad_lines="skip")
# # ───────────────────────────────────────────────────────
# COLS = ["search_keyword", "place_name", "address", "date", "review_text"]
# with open(FILE, "r", encoding="utf-8", errors="replace") as f:
#     df = pd.read_csv(
#         f,
#         names=COLS,
#         header=0,
#         engine="python",
#         quoting=csv.QUOTE_MINIMAL,
#         on_bad_lines="skip",
#     )


# # ───────────────────────────────────────────────────────
# # 3. 전처리 – 리뷰 본문 정규화(공백·대소문자 통일)
# # ───────────────────────────────────────────────────────
# df["review_text_norm"] = (
#     df["review_text"]
#     .astype(str)
#     .str.strip()
#     .str.lower()
#     .str.replace(r"\s+", " ", regex=True)   # 연속 공백 → 1칸
# )

# # ───────────────────────────────────────────────────────
# # 4. 중복 제거
# #    place_name + date + review_text_norm 기준
# # ───────────────────────────────────────────────────────
# KEY_COLS = ["place_name", "date", "review_text_norm"]
# dup_mask   = df.duplicated(subset=KEY_COLS, keep="first")
# num_dups   = dup_mask.sum()
# print(f"중복 제거: {num_dups}행 삭제")

# df_clean = df.loc[~dup_mask].drop(columns=["review_text_norm"])

# # ───────────────────────────────────────────────────────
# # 5. 결과 저장 – CSV 깨지지 않도록 QUOTE_ALL
# # ───────────────────────────────────────────────────────
# OUT = FILE.with_stem(FILE.stem)
# df_clean.to_csv(
#     OUT,
#     index=False,
#     encoding="utf-8-sig",
#     quoting=csv.QUOTE_ALL,
# )
# print("완료 →", OUT)